In [2]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain langchain-community
%pip install -Uq python_dotenv
%pip install -Uq sentence-transformers
%pip install -Uq supabase

You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart t

In [3]:
from supabase import create_client
from dotenv import load_dotenv
import os
load_dotenv()

SUPABASE_URL=os.getenv("SUPABASE_URL")
SUPABASE_KEY=os.getenv("SUPABASE_KEY")

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [4]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage


/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "./docs/constitution.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case


📄 Partitioning document: ./docs/constitution.pdf


The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅ Extracted 3845 elements


In [6]:
elements

 ...]

In [7]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [8]:
len(elements)

3845

In [11]:
elements[0].to_dict()

{'type': 'Image',
 'element_id': '7c15ff0d9f542251c1dd189608f5f6cc',
 'text': '                                ',
 'metadata': {'coordinates': {'points': ((np.float64(668.6111111111111),
     np.float64(187.5)),
    (np.float64(668.6111111111111), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(187.5))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-05-13T23:42:41',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'constitution.pdf'}}

In [13]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

Found 15 images


{'type': 'Image',
 'element_id': '7c15ff0d9f542251c1dd189608f5f6cc',
 'text': '                                ',
 'metadata': {'coordinates': {'points': ((np.float64(668.6111111111111),
     np.float64(187.5)),
    (np.float64(668.6111111111111), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(187.5))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-05-13T23:42:41',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'constitution.pdf'}}

In [14]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 

Found 28 tables


{'type': 'Table',
 'element_id': '78e327fe483387b18e5d258bb4de4da6',
 'text': '1. The Republic and its territories .................................................................... 3 2. Islam to be State religion ............................................................................... 3 2A. The Objectives Resolution to form part of substantive provisions ............ 3 3. Elimination of exploitation ............................................................................ 4 4. Right of individuals to be dealt with in accordance with law, etc. ............. 4 5. Loyalty to State and obedience to Constitution and law ............................. 4 6. High treason .................................................................................................... 4 7. Definition of the State..................................................................................... 6 8. Laws inconsistent with or in derogation of Fundamental Rights to be void .......................

In [15]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 245 chunks


In [16]:
# View all chunks
chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.TableChunk'>"}

In [20]:
# View a single chunk
# chunks[5].to_dict()

# View original elements
chunks[11].metadata.orig_elements[-1].to_dict()

{'type': 'UncategorizedText',
 'element_id': 'fc257f981fcc5b2bcb765acc2571ab68',
 'text': 'CHAPTER 2. – THE SUPREME COURT OF PAKISTAN ........................................... 96',
 'metadata': {'coordinates': {'points': ((np.float64(350.0555555555555),
     np.float64(493.7222222222219)),
    (np.float64(350.0555555555555), np.float64(525.2158888888886)),
    (np.float64(1355.875111111111), np.float64(525.2158888888886)),
    (np.float64(1355.875111111111), np.float64(493.7222222222219))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-05-13T23:42:41',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 8,
  'parent_id': 'b2f8e16ad1d13838a9513cf61cde14e0',
  'file_directory': './docs',
  'filename': 'constitution.pdf'}}